# 03 — Gold: fatos, deltas e superfícies de consumo

Materializa o que exige janela sobre o histórico. As views de diagnóstico mais simples ficam em `fabric/sql/gold_views.sql`, como T-SQL sobre o SQL analytics endpoint.

**A divisão é intencional:** o que precisa de `LAG`/`LEAD` sobre a série inteira é calculado aqui em Spark e materializado; o que é filtro sobre o snapshot atual vira view T-SQL, mais barata de manter e direta para o Power BI.

## As três guardas dos deltas

Delta é o que entrega o valor de "monitoramento contínuo", e também o que mais facilmente vira alarme que ninguém lê.

1. **Dois snapshots consecutivos são obrigatórios.** O primeiro snapshot da série nunca gera delta — sem isso o dia 1 marcaria 3.472 tags como criadas e o dashboard nasceria desacreditado.
2. **Lacuna fica visível.** Intervalo maior que `MAX_GAP_HOURS` marca `delta_status = 'gap'` em vez de ser descartado. Um fim de semana com o PAD parado aparece como lacuna, não como buraco silencioso.
3. **Remoção exige confirmação dupla.** Tag ausente em um snapshot é `candidate`; ausente em dois consecutivos é `confirmed`. Se reaparecer, vira `false_positive` — sinal de falha de coleta, não de remoção.

> **Só `confirmed` deve alertar.** Delete de PI Point apaga o histórico e é irreversível.

In [ ]:
SILVER = "silver_"
GOLD = "gold_"

MAX_GAP_HOURS = 48       # acima disso o par de snapshots é marcado como 'gap'
DEGRADATION_FACTOR = 3.0  # duração acima de N x a média móvel indica degradação de coleta


def run(sql):
    spark.sql(sql)


def report(name):
    print(f"{name:34} {spark.table(name).count():>9,} linhas")

## Pares de snapshots consecutivos

Tabela base de todos os deltas. `delta_status` vale `first` (sem par anterior), `gap` (intervalo grande demais) ou `ok`.

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {GOLD}snapshot_pair AS
WITH snaps AS (
    SELECT snapshot_date, MAX(snapshot_ts) AS snapshot_ts
    FROM {SILVER}dim_pipoint
    GROUP BY snapshot_date
),
seq AS (
    SELECT
        snapshot_date,
        snapshot_ts,
        LAG(snapshot_date)  OVER (ORDER BY snapshot_date) AS prev_snapshot_date,
        LAG(snapshot_ts)    OVER (ORDER BY snapshot_date) AS prev_snapshot_ts,
        LEAD(snapshot_date) OVER (ORDER BY snapshot_date) AS next_snapshot_date
    FROM snaps
)
SELECT
    *,
    (UNIX_TIMESTAMP(snapshot_ts) - UNIX_TIMESTAMP(prev_snapshot_ts)) / 3600.0 AS gap_hours,
    CASE
        WHEN prev_snapshot_date IS NULL THEN 'first'
        WHEN (UNIX_TIMESTAMP(snapshot_ts) - UNIX_TIMESTAMP(prev_snapshot_ts)) / 3600.0 > {MAX_GAP_HOURS}
            THEN 'gap'
        ELSE 'ok'
    END AS delta_status
FROM seq
""")

spark.table(f"{GOLD}snapshot_pair").orderBy("snapshot_date").show(truncate=False)

## Delta 1 — tag criada

Tag que apareceu no Data Archive sem passar pelo processo catalogado.

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {GOLD}delta_tag_criada AS
SELECT
    s.snapshot_date,
    s.prev_snapshot_date,
    s.delta_status,
    s.gap_hours,
    cur.point_id,
    cur.name        AS tag_name,
    cur.point_source,
    cur.point_type,
    cur.descriptor,
    cur.creation_date,
    cur.creator
FROM {GOLD}snapshot_pair s
JOIN {SILVER}dim_pipoint cur
  ON cur.snapshot_date = s.snapshot_date
LEFT JOIN {SILVER}dim_pipoint prv
  ON prv.snapshot_date = s.prev_snapshot_date
 AND prv.point_id      = cur.point_id
WHERE s.delta_status <> 'first'
  AND prv.point_id IS NULL
""")

report(f"{GOLD}delta_tag_criada")

## Delta 2 — tag removida (alerta forte)

| `removal_status` | Significado | Alerta? |
|---|---|---|
| `confirmed` | ausente em dois snapshots consecutivos | **sim** |
| `candidate` | ausente no snapshot mais recente, ainda sem confirmação | não — aguardar o próximo ciclo |
| `false_positive` | sumiu e voltou: foi falha de coleta | não — mas investigue o pipeline |

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {GOLD}delta_tag_removida AS
WITH missing AS (
    SELECT
        s.snapshot_date,
        s.prev_snapshot_date,
        s.next_snapshot_date,
        s.delta_status,
        s.gap_hours,
        prv.point_id,
        prv.name        AS tag_name,
        prv.point_source,
        prv.point_type,
        prv.descriptor,
        prv.creation_date
    FROM {GOLD}snapshot_pair s
    JOIN {SILVER}dim_pipoint prv
      ON prv.snapshot_date = s.prev_snapshot_date
    LEFT JOIN {SILVER}dim_pipoint cur
      ON cur.snapshot_date = s.snapshot_date
     AND cur.point_id      = prv.point_id
    WHERE s.delta_status <> 'first'
      AND cur.point_id IS NULL
)
SELECT
    m.*,
    CASE
        WHEN m.next_snapshot_date IS NULL THEN 'candidate'
        WHEN nxt.point_id IS NULL         THEN 'confirmed'
        ELSE 'false_positive'
    END AS removal_status
FROM missing m
LEFT JOIN {SILVER}dim_pipoint nxt
  ON nxt.snapshot_date = m.next_snapshot_date
 AND nxt.point_id      = m.point_id
""")

report(f"{GOLD}delta_tag_removida")
spark.sql(f"""
SELECT removal_status, COUNT(*) AS qtd
FROM {GOLD}delta_tag_removida GROUP BY removal_status
""").show()

## Delta 3 — atributo alterado

Mudança estrutural no AF. Separado dos outros dois de propósito: misturar "tag nasceu", "tag morreu" e "config mudou" num alerta só produz algo genérico que ninguém consegue acionar.

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {GOLD}delta_atributo_alterado AS
WITH snaps AS (
    SELECT snapshot_date, MAX(snapshot_ts) AS snapshot_ts
    FROM {SILVER}dim_attribute GROUP BY snapshot_date
),
pairs AS (
    SELECT
        snapshot_date,
        LAG(snapshot_date) OVER (ORDER BY snapshot_date) AS prev_snapshot_date,
        (UNIX_TIMESTAMP(snapshot_ts)
         - UNIX_TIMESTAMP(LAG(snapshot_ts) OVER (ORDER BY snapshot_date))) / 3600.0 AS gap_hours
    FROM snaps
)
SELECT
    p.snapshot_date,
    p.prev_snapshot_date,
    p.gap_hours,
    CASE WHEN p.gap_hours > {MAX_GAP_HOURS} THEN 'gap' ELSE 'ok' END AS delta_status,
    cur.attribute_guid,
    cur.element_path,
    cur.attribute_name,
    prv.config_string   AS config_string_anterior,
    cur.config_string   AS config_string_atual,
    prv.resolved_tag    AS resolved_tag_anterior,
    cur.resolved_tag    AS resolved_tag_atual,
    prv.data_reference  AS data_reference_anterior,
    cur.data_reference  AS data_reference_atual,
    prv.uom             AS uom_anterior,
    cur.uom             AS uom_atual,
    prv.attribute_type  AS tipo_anterior,
    cur.attribute_type  AS tipo_atual,
    prv.dr_status       AS dr_status_anterior,
    cur.dr_status       AS dr_status_atual
FROM pairs p
JOIN {SILVER}dim_attribute cur ON cur.snapshot_date = p.snapshot_date
JOIN {SILVER}dim_attribute prv
  ON prv.snapshot_date  = p.prev_snapshot_date
 AND prv.attribute_guid = cur.attribute_guid
WHERE p.prev_snapshot_date IS NOT NULL
  AND (   NOT (cur.config_string  <=> prv.config_string)
       OR NOT (cur.resolved_tag   <=> prv.resolved_tag)
       OR NOT (cur.data_reference <=> prv.data_reference)
       OR NOT (cur.uom            <=> prv.uom)
       OR NOT (cur.attribute_type <=> prv.attribute_type))
""")

report(f"{GOLD}delta_atributo_alterado")

## Fato de saúde das tags

Junta o estado de cada tag com seus metadados e com a cobertura no AF. `is_orfa` sai daqui: tag sem nenhum atributo do AF apontando para ela.

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {GOLD}fact_tag_health AS
WITH af_cobertura AS (
    SELECT snapshot_date, matched_point_id AS point_id, COUNT(*) AS af_attribute_count
    FROM {SILVER}dim_attribute
    WHERE matched_point_id IS NOT NULL
    GROUP BY snapshot_date, matched_point_id
)
SELECT
    h.snapshot_date,
    h.snapshot_ts,
    h.point_id,
    h.name          AS tag_name,
    h.point_type,
    h.point_source,
    h.snapshot_value,
    h.snapshot_status,
    h.value_time,
    h.is_good,
    h.age_hours,
    h.age_days,
    h.is_stale,
    h.is_pt_created,
    h.has_ever_received_data,
    h.creation_date,
    p.descriptor,
    p.eng_units,
    p.compressing,
    p.creator,
    COALESCE(c.af_attribute_count, 0)     AS af_attribute_count,
    COALESCE(c.af_attribute_count, 0) = 0 AS is_orfa
FROM {SILVER}fact_pipoint_health h
LEFT JOIN {SILVER}dim_pipoint p
       ON p.snapshot_date = h.snapshot_date AND p.point_id = h.point_id
LEFT JOIN af_cobertura c
       ON c.snapshot_date = h.snapshot_date AND c.point_id = h.point_id
""")

report(f"{GOLD}fact_tag_health")

## Superfícies de atributo: pública e completa

`config_string` é o campo que liga AF e PI — mascará-lo destruiria o ativo central do projeto. O controle acontece na **superfície de consumo**, não no dado.

| Objeto | `config_string` cru | Uso |
|---|---|---|
| `gold_attribute_public` | ausente | relatório publicado, embed no SharePoint |
| `gold_attribute_full` | presente | diagnóstico, acesso restrito |

O relatório do Power BI deve consumir **apenas** a versão pública.

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {GOLD}attribute_full AS
SELECT * FROM {SILVER}dim_attribute
""")

# Sem config_string cru: pode conter caminho interno, IP ou string de conexão.
run(f"""
CREATE OR REPLACE TABLE {GOLD}attribute_public AS
SELECT
    snapshot_date, snapshot_ts,
    element_path, element_name, element_template,
    attribute_path, attribute_name, attribute_level,
    description, attribute_type, uom, categories, template_name,
    data_reference, resolved_server, resolved_tag, resolved_point_id,
    is_dynamic_reference, parse_status,
    is_configuration_item, is_hidden, is_excluded,
    attribute_guid, element_guid,
    matched_point_id, matched_point_name, match_method, dr_status, name_id_mismatch,
    has_sensitive_hint
FROM {SILVER}dim_attribute
""")

report(f"{GOLD}attribute_public")
report(f"{GOLD}attribute_full")

## Desempenho da coleta

Duração por dataset ao longo do tempo. É o que revela o dia em que o carregamento em lote parar de funcionar — uma degradação de ordem de grandeza é óbvia na série e quase invisível numa execução isolada.

Na validação em campo, a coleta completa variou entre 140 s e 281 s sem nenhuma mudança de código, com o dataset `analyses` indo de 8 s a 125 s. Espere ruído: `is_degradado` compara contra a média móvel dos últimos 30 runs, não contra um valor fixo.

In [ ]:
TS = "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"

run(f"""
CREATE OR REPLACE TABLE {GOLD}run_performance AS
WITH base AS (
    SELECT
        CAST(snapshot_date AS DATE)         AS snapshot_date,
        run_id, run_type,
        TO_TIMESTAMP(snapshot_ts, '{TS}')   AS snapshot_ts,
        status                              AS run_status,
        CAST(run_duration_ms AS BIGINT)     AS run_duration_ms,
        dataset,
        dataset_status,
        CAST(dataset_row_count AS BIGINT)   AS row_count,
        CAST(dataset_duration_ms AS BIGINT) AS duration_ms,
        dataset_error,
        dataset_metrics_json,
        module_version,
        used_fallback_dir
    FROM bronze_run_manifest
),
com_baseline AS (
    SELECT
        *,
        AVG(duration_ms) OVER (
            PARTITION BY dataset ORDER BY snapshot_ts
            ROWS BETWEEN 30 PRECEDING AND 1 PRECEDING
        ) AS baseline_duration_ms
    FROM base
)
SELECT
    *,
    baseline_duration_ms IS NOT NULL
        AND duration_ms > {DEGRADATION_FACTOR} * baseline_duration_ms AS is_degradado
FROM com_baseline
""")

report(f"{GOLD}run_performance")

print("\nÚltima execução por dataset:")
spark.sql(f"""
SELECT dataset, dataset_status, row_count, duration_ms,
       CAST(baseline_duration_ms AS BIGINT) AS baseline_ms, is_degradado
FROM {GOLD}run_performance
WHERE snapshot_ts = (SELECT MAX(snapshot_ts) FROM {GOLD}run_performance)
ORDER BY duration_ms DESC
""").show(truncate=False)

In [ ]:
# Qualidade do parser de ConfigString ao longo do tempo.
# É a falha mais provável do projeto e a mais silenciosa: se a taxa cair, as views de tag órfã
# e DR quebrado ficam erradas sem dar nenhum sinal por conta própria.
print("resolution_rate por execução:")
spark.sql(f"""
SELECT
    snapshot_date,
    CAST(GET_JSON_OBJECT(dataset_metrics_json, '$.resolution_rate') AS DOUBLE) AS resolution_rate,
    CAST(GET_JSON_OBJECT(dataset_metrics_json, '$.pi_point_dr_count') AS INT)  AS dr_count,
    GET_JSON_OBJECT(dataset_metrics_json, '$.resolution_mode')                 AS modo
FROM {GOLD}run_performance
WHERE dataset = 'attributes'
ORDER BY snapshot_date DESC
""").show(20, truncate=False)

print("Baseline medido em campo: 0.9941. Queda maior que 2 p.p. merece investigação —")
print("procure a amostra de ConfigString no log do PowerShell antes de mexer no AF.")